# Set up your environment

Use Python 3.10–3.14 and a virtual environment. Keep the notebooks and their sibling `code` directory together. You can obtain the repository with Git **or download and extract a repository ZIP**; a Git clone is not required for these lessons.

The [chapter guide](../README.md) has the complete installation commands. Choose the installation matching the notebook edition:

- **Development / before the 2.10 release:** install NVFlare from the same repository checkout or extracted source archive as these notebooks.
- **Published 2.10 edition:** install the 2.10 package from PyPI and use the matching release's notebooks.
- **Original 2.6 course:** use its preserved notebooks and matching environment rather than mixing editions.

In a terminal at the repository root:

```bash
python3 -m venv .venv-chapter1
source .venv-chapter1/bin/activate
python -m pip install --upgrade pip
```

Choose **one** FLARE install command:

```bash
# Development source (works from a checkout or an extracted source archive):
NVFL_BASE_VERSION=2.10.0 python -m pip install -e '.[pt]'

# Published 2.10 edition, once available on PyPI:
# python -m pip install 'nvflare[pt]>=2.10.0,<2.11'
```

`NVFL_BASE_VERSION` supplies package metadata for an untagged development ZIP; it does not change which source code is installed. Then install and start Jupyter from this same environment:

```bash
python -m pip install jupyterlab ipykernel
python -m ipykernel install --user --name nvflare-chapter1 --display-name 'NVFlare Self-Paced Training'
python -m jupyterlab
```

Open this notebook and select **NVFlare Self-Paced Training** as the kernel. The `%pip` command in later optional lessons installs into that selected kernel; terminal commands must use the same environment.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version

import nvflare
import torch
from packaging.version import Version
from nvflare.recipe import SimEnv, add_final_global_evaluation, set_per_site_config

installed = {}
for package in ("nvflare", "nvflare-nightly"):
    try:
        installed[package] = version(package)
    except PackageNotFoundError:
        pass
assert len(installed) == 1, "Use a fresh environment with one FLARE distribution installed"
package, package_version = next(iter(installed.items()))
print("Python:", sys.executable)
print("Installed FLARE package:", package, package_version)
print("Imported source:", nvflare.__file__)
print("PyTorch:", torch.__version__)
assert (3, 10) <= sys.version_info[:2] <= (3, 14)
assert Version(package_version).release[:2] == (2, 10), "Use the environment for this 2.10 chapter"


In [ ]:
from pathlib import Path
from IPython.display import Code, display

chapter = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/job.py").is_file())
code = chapter / "code"
print("Run the terminal commands from:", code)

For a source ZIP, `nvflare.__version__` can be `0+unknown`; the check above uses the installed package metadata and verifies that the required Recipe helpers import.

## Understand the data before training

Each site generates **200 training images and 100 evaluation images**, each with shape `(3, 32, 32)`, across 10 classes. A bright patch encodes the class; low-amplitude noise differs by site and split. The generator derives stable seeds from the site name and split, so repeated runs see the same data.

There is no shared training dataset and no network download. Train and evaluation samples are distinct. The two sites have the same class distribution (IID); learning with different site distributions is a later exercise.

The model is a small CNN, trained on CPU. Default training is one local epoch per round, batch size 32, learning rate 0.1, and three federated rounds. With 200 samples, a client performs **7 optimizer steps per round**, including the final partial batch.

In [ ]:
display(Code(filename=str(code / "data.py"), language="python"))

**Check yourself:** Why must the evaluation seed differ from the training seed? Why is 200 divided by 32 rounded up when counting optimizer steps?

If an import fails, compare `sys.executable` above with `python -c "import sys; print(sys.executable)"` in the terminal. Select the correct kernel or activate the correct environment before installing more packages.

Continue to [run the job](running_pytorch_fl_job.ipynb).